In [1]:
import os
import csv
import glob

In [2]:
#create csv database for nii files, 2 columns : T1 file path, T2 file path
nii_base = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/xnat_downloads"
# Recorrer todos los sujetos dentro de nii_base y emparejar archivos T1/T2 por sujeto
import re
import csv

def _normalize_key(filename):
    # Elimino tokens T1/T2 (insensible a mayúsculas) y normalizo separadores para agrupar series similares
    name = re.sub(r'(?i)t1|t2', '', filename)
    name = re.sub(r'\.nii(\.gz)?$', '', name, flags=re.IGNORECASE)
    # sustituir múltiples espacios/underscores por un único underscore
    name = re.sub(r'[\s_]+', '_', name).strip('_').lower()
    return name

def create_database_csv(nii_base, output_csv):
    """
    Crea un CSV con columnas: subject,T1_path,T2_path
    - Recorre recursivamente cada carpeta de sujeto dentro de `nii_base`
    - Detecta archivos .nii y .nii.gz
    - Determina si un archivo es T1 o T2 buscando 'T1' o 'T2' en el nombre (insensible a mayúsculas)
    - Empareja archivos por un nombre normalizado (quitando el token T1/T2)
    - Escribe una fila por pareja encontrada; si falta T1 o T2, la columna quedará vacía y se imprimirá una advertencia
    """
    subjects = [d for d in os.listdir(nii_base) if os.path.isdir(os.path.join(nii_base, d))]
    print(subjects)
    rows = []
    for subj in sorted(subjects):
        subj_dir = os.path.join(nii_base, subj)
        # mapa: normalized_key -> { 't1': path, 't2': path }
        pairs = {}
        for root, _, files in os.walk(subj_dir):
            for fname in files:
                if not (fname.lower().endswith('.nii') or fname.lower().endswith('.nii.gz')):
                    continue
                fl = fname.lower()
                fullpath = os.path.join(root, fname)
                if 't1' in fl:
                    key = _normalize_key(fname)
                    print(key)
                    pairs.setdefault(key, {})['t1'] = fullpath
                elif 't2' in fl:
                    key = _normalize_key(fname)
                    pairs.setdefault(key, {})['t2'] = fullpath
        # convertir pairs en filas (subject, t1, t2)
        for key, entry in pairs.items():
            # print(key)
            # print(entry)
            t1 = entry.get('t1', '')
            t2 = entry.get('t2', '')
            if (not t1) and (not t2) and entry.get('other'):
                # si sólo hay 'other', ponemos la primera en la columna T1 para revisión
                t1 = entry['other'][0]
                print(f'Warning: subject {subj} - file(s) found without T1/T2 token for key "{key}": {entry[other]}. Saved as T1 for inspection.')
            if not t1 or not t2:
                print(f'Warning: subject {subj} - incomplete pair for key "{key}": T1={t1}, T2={t2}')
            rows.append((subj, t1, t2))
    # escribir CSV
    # os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    # with open(output_csv, 'w', newline='') as csvfile:
    #     writer = csv.writer(csvfile)
    #     writer.writerow(['subject', 'T1_path', 'T2_path'])
    #     for r in rows:
    #         writer.writerow(r)
    # print(f'Wrote {len(rows)} rows to {output_csv}')

# Ejemplo de uso:
# create_database_csv(nii_base, '/path/to/output/xnat_pairs.csv')


In [38]:
create_database_csv(nii_base, "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/model_segmentation/output.csv")

['XNAT_E99161', 'XNAT_E78069', 'XNAT_E78225', 'XNAT_E77497', 'XNAT_E98902', 'XNAT_E77463', 'XNAT_E86321', 'XNAT_E86958', 'XNAT_E77363', 'XNAT_E126255', 'XNAT_E86818', 'XNAT_E98944', 'XNAT_E77840', 'XNAT_E86970', 'XNAT_E78329', 'XNAT_E98831', 'XNAT_E78165', 'XNAT_E77994', 'XNAT_E77995', 'XNAT_E86447', 'XNAT_E89951', 'XNAT_E90517', 'XNAT_E77876', 'XNAT_E99116', 'XNAT_E77913', 'XNAT_E77632', 'XNAT_E78205', 'XNAT_E99181', 'XNAT_E77786', 'XNAT_E77891', 'XNAT_E78149', 'XNAT_E77940', 'XNAT_E86916', 'XNAT_E78306', 'XNAT_E77501', 'XNAT_E105052', 'XNAT_E90490', 'XNAT_E77354', 'XNAT_E77493', 'XNAT_E125915', 'XNAT_E78086', 'XNAT_E78314', 'XNAT_E86972', 'XNAT_E86559', 'XNAT_E104345', 'XNAT_E86941', 'XNAT_E77920', 'XNAT_E99238', 'XNAT_E77625', 'XNAT_E78311', 'XNAT_E86932', 'XNAT_E77506', 'XNAT_E77966', 'XNAT_E86824', 'XNAT_E77416', 'XNAT_E99048', 'XNAT_E90402', 'XNAT_E77825', 'XNAT_E78134', 'XNAT_E77347', 'XNAT_E77568', 'XNAT_E77662', 'XNAT_E98791', 'XNAT_E78203', 'XNAT_E77977', 'XNAT_E86299', 'XNAT

In [3]:
#create small csv database for testing
def create_test_database_csv(output_csv):
    test_rows = [
        ('Ceib','S0','E0','/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/xnat_downloads/outputs/XNAT_E77363/XNAT_E77363/6/SAG T1 LUMBAR SAG T1 LUMBAR 6 6_SAG T1 LUMBAR SAG T1 LUMBAR 6 6.nii.gz', '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/xnat_downloads/outputs/XNAT_E77363/XNAT_E77363/4/SAG T2 LUMBAR SAG T2 LUMBAR 4 4_SAG T2 LUMBAR SAG T2 LUMBAR 4 4.nii.gz'),
        ('Ceib','S1','E1', '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/xnat_downloads/outputs/XNAT_E77463/XNAT_E77463/3/SAG T1 LUMBAR SAG T1 LUMBAR 3 3_SAG T1 LUMBAR SAG T1 LUMBAR 3 3.nii.gz', '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/xnat_downloads/outputs/XNAT_E77463/XNAT_E77463/2/SAG T2 LUMBAR SAG T2 LUMBAR 2 2_SAG T2 LUMBAR SAG T2 LUMBAR 2 2.nii.gz')
    ]
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['proj', 'subj', 'sess', 'T2w', 'T1w', 'Stir', '/derivatives/manualSeg/', 'T2w_bfcorr', 'T1w_bfcorr', 'T1w_reg'])
        for r in test_rows:
            writer.writerow(r)
    print(f'Wrote test CSV with {len(test_rows)} rows to {output_csv}')

In [4]:
create_test_database_csv("/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/model_segmentation/input_data.csv")

Wrote test CSV with 2 rows to /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/model_segmentation/input_data.csv


## change subfolders


In [ ]:
import os
import shutil

ROOT = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib"

# OPTIONS:
MODE = "copy"  # "copy" or "move"
DRY_RUN = False  # set True if you only want to see what would happen

def find_nii_files(patient_root):
    nii_files = []
    for dirpath, dirnames, filenames in os.walk(patient_root):
        for f in filenames:
            if f.lower().endswith(".nii.gz"):
                nii_files.append(os.path.join(dirpath, f))

    return nii_files

def classify_t1_t2(nii_files):
    t1_candidates = []
    t2_candidates = []
    for path in nii_files:
        name = os.path.basename(path).lower()
        if "t1" in name:
            t1_candidates.append(path)
        if "t2" in name:
            t2_candidates.append(path)
    t1 = sorted(t1_candidates)[0] if t1_candidates else None
    t2 = sorted(t2_candidates)[0] if t2_candidates else None
    return t1, t2

def do_transfer(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if MODE == "copy":
        shutil.copy2(src, dst)
    elif MODE == "move":
        shutil.move(src, dst)
    else:
        raise ValueError(f"Unknown MODE: {MODE}")

def main():
    patients = sorted(
        d for d in os.listdir(ROOT)
        if os.path.isdir(os.path.join(ROOT, d))
    )

    if not patients:
        print("No patient folders found under:", ROOT)
        return

    print(f"Found {len(patients)} patient folders under {ROOT}")
    print(f"MODE = {MODE}, DRY_RUN = {DRY_RUN}")

    for idx, pat_dir in enumerate(patients, start=1):
        pat_root = os.path.join(ROOT, pat_dir)
        subj_id = f"sub-S{idx:02d}"
        ses_id = f"ses-E{idx:02d}"
        new_anat_dir = os.path.join(ROOT, subj_id, ses_id, "mod-mr", "anat")

        print(f"\nPatient folder: {pat_dir}")
        print(f"  -> {pat_root}")
        nii_files = find_nii_files(pat_root)
        print(f"  Found {len(nii_files)} .nii.gz files")

        if not nii_files:
            print("  WARNING: no .nii.gz files found, skipping")
            continue

        t1_path, t2_path = classify_t1_t2(nii_files)

        if not t1_path and not t2_path:
            print("  WARNING: no files with T1/T2 in the name, skipping")
            continue

        if t1_path:
            new_t1_name = f"{subj_id}_{ses_id}_Sag_T1.nii.gz"
            dest_t1 = os.path.join(new_anat_dir, new_t1_name)
            print(f"  T1: {t1_path} -> {dest_t1}")
        else:
            dest_t1 = None
            print("  WARNING: no T1 file found")

        if t2_path:
            new_t2_name = f"{subj_id}_{ses_id}_Sag_T2.nii.gz"
            dest_t2 = os.path.join(new_anat_dir, new_t2_name)
            print(f"  T2: {t2_path} -> {dest_t2}")
        else:
            dest_t2 = None
            print("  WARNING: no T2 file found")

        if DRY_RUN:
            continue

        # Actually copy/move
        if t1_path and dest_t1:
            do_transfer(t1_path, dest_t1)
        if t2_path and dest_t2:
            do_transfer(t2_path, dest_t2)

if __name__ == "__main__":
    main()


Found 332 patient folders under /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib
MODE = copy, DRY_RUN = False

Patient folder: XNAT_E104345
  -> /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/XNAT_E104345
  Found 2 .nii.gz files
  T1: /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/XNAT_E104345/XNAT_E104345/1403/T1W_TSE GD T1W_TSE GD 1403 1403_T1W_TSE GD T1W_TSE GD 1403 1403.nii.gz -> /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/sub-S01/ses-E01/mod-mr/anat/sub-S01_ses-E01_Sag_T1.nii.gz
  T2: /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/XNAT_E104345/XNAT_E104345/603/T2W_TSE T2W_TSE 603 603_T2W_TSE T2W_TSE 603 603.nii.gz -> /mnt/datalake/openmind/Midas/

In [7]:
import re
import pandas as pd

def generate_csv_from_log(log_file_path, output_csv_name):
    # 1. Leer el contenido del archivo de texto
    try:
        with open(log_file_path, 'r') as file:
            content = file.read()
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {log_file_path}")
        return

    # Lista para guardar los datos de cada paciente
    data = []

    # 2. Dividir el contenido en bloques por paciente
    # Usamos "Patient folder:" como separador
    # El [1:] es para saltar el primer fragmento antes del primer paciente
    blocks = content.split('Patient folder: ')[1:] 

    for block in blocks:
        lines = block.split('\n')
        
        # La primera línea del bloque contiene el ID de XNAT (ej. XNAT_E104345)
        xnat_id = lines[0].strip()
        
        # Variables temporales para este paciente
        subj = None
        sess = None
        t1_file = None
        t2_file = None
        
        # Analizar las líneas para buscar las rutas de los archivos transformados
        for line in lines:
            if '->' in line:
                # Separar origen y destino
                parts = line.split('->')
                dest_path = parts[1].strip()
                filename = dest_path.split('/')[-1] # Nombre del archivo final
                
                # Usamos Expresiones Regulares (Regex) para extraer Subject y Session
                # Busca patrones como 'sub-S01' y 'ses-E01'
                match = re.search(r'(sub-S\d+)_(ses-E\d+)_', filename)
                if match:
                    # Extraemos solo el código (quitamos el prefijo 'sub-' y 'ses-')
                    subj = match.group(1).replace('sub-', '')
                    sess = match.group(2).replace('ses-', '')
                
                # Identificar si es T1 o T2 basado en el nombre del archivo
                if 'Sag_T1' in filename or '_T1.' in filename:
                    t1_file = filename
                elif 'Sag_T2' in filename or '_T2.' in filename:
                    t2_file = filename
        
        # Si encontramos datos válidos, los agregamos a la lista
        if subj and sess:
            data.append({
                'proj': 'Ceib',
                'subj': subj,
                'sess': sess,
                'T2w': t2_file if t2_file else '',
                'T1w': t1_file if t1_file else '',
                'Stir': '',
                '/derivatives/manualSeg/': '',
                'T2w_bfcorr': '',
                'T1w_bfcorr': '',
                'T1w_reg': '',
                'XNAT_name': xnat_id  # <--- Aquí agregamos la columna nueva
            })

    # 3. Crear el DataFrame y guardar el CSV
    if data:
        df = pd.DataFrame(data)
        
        # Reordenar columnas si es necesario para asegurar el formato exacto
        cols = ['proj', 'subj', 'sess', 'T2w', 'T1w', 'Stir', 
                '/derivatives/manualSeg/', 'T2w_bfcorr', 'T1w_bfcorr', 'T1w_reg', 'XNAT_name']
        df = df[cols]
        
        df.to_csv(output_csv_name, index=False)
        print(f"¡Éxito! Archivo '{output_csv_name}' creado con {len(df)} filas.")
        print(df.head()) # Muestra las primeras filas para verificar
    else:
        print("No se encontraron datos para procesar.")

# --- Ejecución del script ---
if __name__ == "__main__":
    generate_csv_from_log('/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/names.txt', 'input_data_with_xnat.csv')

¡Éxito! Archivo 'input_data_with_xnat.csv' creado con 328 filas.
   proj subj sess                            T2w  \
0  Ceib  S01  E01  sub-S01_ses-E01_Sag_T2.nii.gz   
1  Ceib  S02  E02  sub-S02_ses-E02_Sag_T2.nii.gz   
2  Ceib  S03  E03  sub-S03_ses-E03_Sag_T2.nii.gz   
3  Ceib  S04  E04  sub-S04_ses-E04_Sag_T2.nii.gz   
4  Ceib  S05  E05  sub-S05_ses-E05_Sag_T2.nii.gz   

                             T1w Stir /derivatives/manualSeg/ T2w_bfcorr  \
0  sub-S01_ses-E01_Sag_T1.nii.gz                                           
1  sub-S02_ses-E02_Sag_T1.nii.gz                                           
2  sub-S03_ses-E03_Sag_T1.nii.gz                                           
3  sub-S04_ses-E04_Sag_T1.nii.gz                                           
4  sub-S05_ses-E05_Sag_T1.nii.gz                                           

  T1w_bfcorr T1w_reg     XNAT_name  
0                     XNAT_E104345  
1                     XNAT_E104415  
2                     XNAT_E104484  
3                

In [9]:
import pandas as pd
import re

def procesar_csv_corregido(input_csv, output_csv):
    try:
        # 1. Cargar el CSV solucionando el error de encoding y el separador
        # 'latin-1' o 'cp1252' suelen arreglar el error del byte 0xa0
        # 'sep=;' le dice a pandas que use punto y coma en vez de coma
        df = pd.read_csv(input_csv, sep=';', encoding='latin-1')
        
        # Limpieza opcional: eliminar espacios en blanco en los nombres de las columnas
        df.columns = df.columns.str.strip()

        # 2. Función para extraer el ID
        def extract_id(url):
            if pd.isna(url) or not isinstance(url, str):
                return None
            # Busca 'experimentId=' seguido de 'XNAT_E' y alfanuméricos
            match = re.search(r'experimentId=(XNAT_E\w+)', url)
            if match:
                return match.group(1)
            return None

        # 3. Buscar la columna correcta (a veces Python distingue mayúsculas)
        col_url = None
        if 'viwer_url' in df.columns:
            col_url = 'viwer_url'
        elif 'Viwer_url' in df.columns:
            col_url = 'Viwer_url'
        
        if col_url:
            print(f"Columna de URL encontrada: {col_url}")
            # Aplicar la extracción
            df['XNAT_name'] = df[col_url].apply(extract_id)
            
            # 4. Guardar el resultado
            # Usamos el mismo encoding y separador para mantener el formato original
            df.to_csv(output_csv, index=False, sep=';', encoding='latin-1')
            
            print(f"¡Éxito! Archivo guardado en: {output_csv}")
            print(df[['XNAT_name', col_url]].head())
        else:
            print(f"Error: No se encontró la columna 'viwer_url'. Columnas disponibles: {list(df.columns)}")

    except Exception as e:
        print(f"Error crítico: {e}")

# --- Ejecutar ---
# --- Ejecución ---
# Reemplaza con el nombre real de tu archivo
procesar_csv_corregido('/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/Patients_images.csv', '/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/Patients_images_xnat.csv')

Columna de URL encontrada: viwer_url
¡Éxito! Archivo guardado en: /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/Patients_images_xnat.csv
     XNAT_name                                          viwer_url
0  XNAT_E77256  https://ceib.cipf.es/xnat2/VIEWER/?subjectId=X...
1  XNAT_E77257  https://ceib.cipf.es/xnat2/VIEWER/?subjectId=X...
2  XNAT_E77264  https://ceib.cipf.es/xnat2/VIEWER/?subjectId=X...
3  XNAT_E77266  https://ceib.cipf.es/xnat2/VIEWER/?subjectId=X...
4  XNAT_E77282  https://ceib.cipf.es/xnat2/VIEWER/?subjectId=X...


In [10]:
import pandas as pd

def merge_patient_measurements(project_csv, measurements_csv, output_csv):
    try:
        # 1. Cargar datos del proyecto (input_data_with_xnat.csv)
        # Separador estándar ','
        df_project = pd.read_csv(project_csv)
        
        # 2. Cargar mediciones (output_data.csv)
        # Separador ';' y codificación 'latin-1' (según vimos antes)
        df_measurements = pd.read_csv(measurements_csv, sep=';', encoding='latin-1')
        
        # 3. Limpieza de claves (XNAT_name)
        # Aseguramos que sean texto y quitamos espacios extraños
        df_project['XNAT_name'] = df_project['XNAT_name'].astype(str).str.strip()
        df_measurements['XNAT_name'] = df_measurements['XNAT_name'].astype(str).str.strip()
        
        # 4. Seleccionar columnas a añadir
        # Solo nos quedamos con el ID y las columnas nuevas para no duplicar basura
        cols_to_add = ['L5-S', 'L4-L5', 'L3-L4', 'L2-L3', 'L1-L2']
        
        # Verificar que existan en el archivo de mediciones
        available_cols = [c for c in cols_to_add if c in df_measurements.columns]
        if not available_cols:
            print("Error: No se encontraron las columnas de mediciones (L5-S, etc) en el CSV.")
            return

        subset_measurements = df_measurements[['XNAT_name'] + available_cols]
        
        # Eliminamos duplicados en las mediciones por si acaso (un paciente = una fila)
        subset_measurements = subset_measurements.drop_duplicates(subset=['XNAT_name'])

        # 5. Fusión (Merge)
        # Usamos 'left' join para mantener todos los pacientes del proyecto original
        # y solo rellenar datos donde haya coincidencia.
        df_final = pd.merge(df_project, subset_measurements, on='XNAT_name', how='left')
        
        # 6. Guardar
        df_final.to_csv(output_csv, index=False)
        print(f"¡Proceso completado! Archivo guardado: {output_csv}")
        print(f"Total pacientes: {len(df_final)}")
        
        # Muestra un ejemplo de filas que sí encontraron match (que no tengan NaN en L5-S)
        match_sample = df_final[df_final['L5-S'].notna()].head()
        if not match_sample.empty:
            print("\nEjemplo de datos cruzados correctamente:")
            print(match_sample[['subj', 'XNAT_name'] + available_cols])
        else:
            print("\nAdvertencia: No se encontraron coincidencias de 'XNAT_name' entre los archivos.")

    except FileNotFoundError as e:
        print(f"Error: No se encontró el archivo. {e}")
    except Exception as e:
        print(f"Ocurrió un error inesperado: {e}")

# --- Ejecución ---
# Asegúrate de usar los nombres reales de tus archivos aquí:
merge_patient_measurements('/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/input_data_with_xnat.csv', '/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/Patients_images_xnat.csv', 'final_merged_data.csv')

¡Proceso completado! Archivo guardado: final_merged_data.csv
Total pacientes: 328

Ejemplo de datos cruzados correctamente:
  subj     XNAT_name  L5-S  L4-L5 L3-L4  L2-L3  L1-L2
0  S01  XNAT_E104345   3.0    1.0     1    2.0    NaN
1  S02  XNAT_E104415   3.0    2.0     2    2.0    2.0
2  S03  XNAT_E104484   4.0    1.0     1    1.0    1.0
3  S04  XNAT_E104616   5.0    3.0     2    2.0    2.0
4  S05  XNAT_E104717   3.0    2.0     2    2.0    2.0


In [1]:
import csv

# Output filename
filename = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/subjects.csv"

# Header for CSV
header = [
    "proj", "subj", "sess", "T2w", "T1w", "Stir",
    "/derivatives/manualSeg/", "T2w_bfcorr", "T1w_bfcorr", "T1w_reg"
]

# Create rows from 1 to 331
rows = []
for i in range(1, 332):
    subj_num = f"S{i:02d}"
    sess_num = f"E{i:02d}"
    row = [
        "Ceib",
        subj_num,
        sess_num,
        f"sub-{subj_num}_ses-{sess_num}_Sag_T1.nii.gz",
        f"sub-{subj_num}_ses-{sess_num}_Sag_T2.nii.gz",
        "",  # Stir (empty as in your example; you can fill if needed)
        "",
        "",
        "",
        ""
    ]
    rows.append(row)

# Write to CSV
with open(filename, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(rows)

print(f"✅ CSV file '{filename}' generated successfully with 331 rows.")



✅ CSV file '/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/subjects.csv' generated successfully with 331 rows.


In [2]:
import pandas as pd
import os

def create_long_format_csv(input_csv_path, output_csv_path):
    # 1. Define Base Paths
    base_t1_t2 = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib"
    base_discs = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/cropped_discs_aligned"

    # 2. Load the input CSV
    try:
        df = pd.read_csv(input_csv_path)
    except FileNotFoundError:
        print(f"Error: Input file '{input_csv_path}' not found.")
        return

    # List to collect all the new rows
    new_rows = []

    # 3. Define the Mapping
    # Maps the column name in your CSV to the suffix used in the filename
    # Note: 'L5-S' column maps to 'L5-S1' filename
    disc_map = {
        'L1-L2': 'L1-L2',
        'L2-L3': 'L2-L3',
        'L3-L4': 'L3-L4',
        'L4-L5': 'L4-L5',
        'L5-S':  'L5-S1' 
    }

    # 4. Iterate through each patient row
    for index, row in df.iterrows():
        subj_id = str(row['subj']).strip()  # e.g., "S01"
        sess_id = str(row['sess']).strip()  # e.g., "E01"
        xnat_name = row['XNAT_name']

        # Construct folder strings ensuring "sub-" and "ses-" prefixes exist
        # This handles cases whether your CSV has "S01" or "sub-S01"
        sub_folder = subj_id if subj_id.startswith("sub-") else f"sub-{subj_id}"
        ses_folder = sess_id if sess_id.startswith("ses-") else f"ses-{sess_id}"
        
        # Used for filenames (e.g. sub-S01_ses-E01...)
        subj_clean = sub_folder
        sess_clean = ses_folder

        # Construct Full Image Paths
        # Pattern: .../Ceib/sub-S02/ses-E02/mod-mr/anat/sub-S02_ses-E02_Sag_T1.nii.gz
        image_t1_path = os.path.join(base_t1_t2, sub_folder, ses_folder, 'mod-mr', 'anat', f"{subj_clean}_{sess_clean}_Sag_T1.nii.gz")
        image_t2_path = os.path.join(base_t1_t2, sub_folder, ses_folder, 'mod-mr', 'anat', f"{subj_clean}_{sess_clean}_Sag_T2.nii.gz")

        # 5. Create a row for each disc type
        for csv_col, file_suffix in disc_map.items():
            # Get the Pfirrmann grade from the specific column
            pfirrmann_grade = row.get(csv_col, None)

            # Construct Disc Path
            # Pattern: .../cropped_discs_aligned/sub-S02/sub-S02_L1-L2.nii.gz
            disc_file_path = os.path.join(base_discs, sub_folder, f"{subj_clean}_{file_suffix}.nii.gz")

            new_row = {
                'subj': subj_id,
                'sess': sess_id,
                'image_T2': image_t2_path,
                'image_T1': image_t1_path,
                'disc': file_suffix,      # The standardized name (e.g. L5-S1)
                'disc_path': disc_file_path,
                'XNAT_name': xnat_name,
                'Pfirrmann': pfirrmann_grade
            }
            new_rows.append(new_row)

    # 6. Create DataFrame and Save
    output_cols = ['subj', 'sess', 'image_T2', 'image_T1', 'disc', 'disc_path', 'XNAT_name', 'Pfirrmann']
    df_long = pd.DataFrame(new_rows, columns=output_cols)

    df_long.to_csv(output_csv_path, index=False)
    print(f"Success! Created '{output_csv_path}' with {len(df_long)} rows.")
    print(df_long.head())

# --- Execution ---
# Replace with your actual file names
create_long_format_csv('/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/final_merged_data.csv', 'final_dataset_long_format.csv')

Success! Created 'final_dataset_long_format.csv' with 1640 rows.
  subj sess                                           image_T2  \
0  S01  E01  /mnt/datalake/openmind/Midas/training_wheels_S...   
1  S01  E01  /mnt/datalake/openmind/Midas/training_wheels_S...   
2  S01  E01  /mnt/datalake/openmind/Midas/training_wheels_S...   
3  S01  E01  /mnt/datalake/openmind/Midas/training_wheels_S...   
4  S01  E01  /mnt/datalake/openmind/Midas/training_wheels_S...   

                                            image_T1   disc  \
0  /mnt/datalake/openmind/Midas/training_wheels_S...  L1-L2   
1  /mnt/datalake/openmind/Midas/training_wheels_S...  L2-L3   
2  /mnt/datalake/openmind/Midas/training_wheels_S...  L3-L4   
3  /mnt/datalake/openmind/Midas/training_wheels_S...  L4-L5   
4  /mnt/datalake/openmind/Midas/training_wheels_S...  L5-S1   

                                           disc_path     XNAT_name Pfirrmann  
0  /mnt/datalake/openmind/Midas/training_wheels_S...  XNAT_E104345       NaN  
1

## create a csv with names and paths

In [9]:
#create a csv with names and paths
# ID,Image,Mask,Subject_MIDS,Session_MIDS,Subject_XNAT,Session_XNAT


CEIB_ROOT = "/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib"
DERIV_AUTSEG_ROOT = os.path.join(CEIB_ROOT, "derivatives", "aut_seg")
OUTPUT_CSV = "midas_ceib_paths.csv"


def main():
    # Find all T2 images in the main Ceib folder (excluding derivatives)
    image_pattern = os.path.join(
        CEIB_ROOT,
        "sub-*",
        "ses-*",
        "mod-mr",
        "anat",
        "sub-*_ses-*_Sag_T2.nii.gz",
    )
    image_files = sorted(glob.glob(image_pattern))

    rows = []
    idx = 1

    for img_path in image_files:
        # Example:
        # /.../Ceib/sub-S01/ses-E01/mod-mr/anat/sub-S01_ses-E01_Sag_T2.nii.gz
        parts = img_path.split(os.sep)

        try:
            sub_index = parts.index("Ceib") + 1  # position of sub-Sxx
        except ValueError:
            # If "Ceib" is not in the path, skip
            continue

        subject = parts[sub_index]      # sub-S01
        session = parts[sub_index + 1]  # ses-E01

        # Build mask path:
        # /.../Ceib/derivatives/aut_seg/sub-S01/ses-E01/mod-mr/anat/sub-S01_ses-E01_Sag_T1_disc.nii.gz
        mask_filename = f"{subject}_{session}_Sag_T1_disc.nii.gz"
        mask_path = os.path.join(
            DERIV_AUTSEG_ROOT,
            subject,
            session,
            "mod-mr",
            "anat",
            mask_filename,
        )

        if not os.path.exists(mask_path):
            # If there is no corresponding mask, skip this case
            # Remove this "continue" if you prefer to keep the row with empty Mask field
            continue

        row = {
            "ID": f"s_{idx}",
            "Image": img_path,
            "Mask": mask_path,
            "Subject_MIDS": subject,   # e.g. "sub-S01"
            "Session_MIDS": session,   # e.g. "ses-E01"
        }
        rows.append(row)
        idx += 1

    # Write CSV
    fieldnames = ["ID", "Image", "Mask", "Subject_MIDS", "Session_MIDS"]
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Written {len(rows)} rows to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()



Written 74 rows to midas_ceib_paths.csv


## Correlation between xnat name and sub-name

In [3]:
import pydicom

def buscar_sesion_xnat(dcm_path):
    try:
        ds = pydicom.dcmread(dcm_path)
        
        print(f"--- Analizando: {dcm_path} ---")
        
        # 1. Accession Number (Candidato #1)
        if 'AccessionNumber' in ds:
            print(f"Accession Number (0008,0050): {ds.AccessionNumber}")
        else:
            print("Accession Number: No encontrado")

        # 2. Study ID (Candidato #2)
        if 'StudyID' in ds:
            print(f"Study ID (0020,0010): {ds.StudyID}")
        
        # 3. Etiquetas Privadas (Si XNAT usó configuración personalizada)
        # XNAT a veces usa etiquetas privadas en el grupo (0029,xxxx)
        # Iteramos para buscar texto que parezca una sesión
        print("\n--- Buscando en etiquetas privadas o comentarios ---")
        
        if 'PatientComments' in ds:
             print(f"Patient Comments: {ds.PatientComments}")

    except Exception as e:
        print(f"Error leyendo el archivo: {e}")

# Ejecutar
buscar_sesion_xnat("/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/xnat_downloads/XNAT_E77256/9/SAG T2 FRFSE SAG T2 FRFSE 9 9/1.2.826.0.1.3680043.10.474.299538.178775-9-1-1hco9rd.dcm")

--- Analizando: /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/xnat_downloads/XNAT_E77256/9/SAG T2 FRFSE SAG T2 FRFSE 9 9/1.2.826.0.1.3680043.10.474.299538.178775-9-1-1hco9rd.dcm ---
Accession Number (0008,0050): 001458
Study ID (0020,0010): 

--- Buscando en etiquetas privadas o comentarios ---
Patient Comments: Project:p0032025  Subject:001421 Session:001458


In [6]:
import nibabel as nib
import numpy as np

def inspeccionar_contenido_nifti(nifti_path):
    # Cargar la imagen
    img = nib.load(nifti_path)
    header = img.header
    
    print(f"=== REPORTE TÉCNICO: {nifti_path} ===\n")

    # 1. Dimensiones (Shape)
    # Ejemplo: (256, 256, 192) es un volumen 3D. 
    # Si ves (256, 256, 192, 100), es un volumen 4D (fMRI o DTI con tiempo/direcciones).
    shape = img.shape
    print(f"1. Dimensiones de la matriz (x, y, z, t): {shape}")
    
    # 2. Tamaño del Vóxel (Zooms)
    # Indica cuánto mide cada píxel en milímetros reales.
    zooms = header.get_zooms()
    print(f"2. Resolución (Tamaño del vóxel en mm): {zooms}")
    
    # 3. Orientación y Posición (Matriz Afín)
    # Esto le dice al visor dónde está la imagen en el espacio físico.
    print(f"3. Matriz Afín (Muestra de la orientación):\n{img.affine[:3, :3]}") # Solo mostramos rotación/zoom
    
    # 4. Tipo de datos
    # float32, int16, etc. Importante para saber la precisión.
    dtype = header.get_data_dtype()
    print(f"4. Tipo de dato numérico: {dtype}")

    # 5. Análisis de la señal (Los píxeles reales)
    # Cargamos los datos en memoria para ver si hay imagen o es ruido
    data = img.get_fdata()
    
    min_val = np.min(data)
    max_val = np.max(data)
    mean_val = np.mean(data)
    
    print(f"\n=== ESTADÍSTICAS DE SEÑAL ===")
    print(f"Intensidad Mínima: {min_val}")
    print(f"Intensidad Máxima: {max_val} (Si es 0, la imagen está negra)")
    print(f"Intensidad Media:  {mean_val:.2f}")

    if len(shape) > 3:
        print(f"Nota: Esta es una imagen 4D con {shape[3]} volúmenes temporales.")


inspeccionar_contenido_nifti('/mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/sub-S02/ses-E02/mod-mr/anat/sub-S02_ses-E02_Sag_T1.nii.gz')

=== REPORTE TÉCNICO: /mnt/datalake/openmind/Midas/training_wheels_Spine/lib/my_lib/segmentation_nets/predict_nii_up/patients_midas/Ceib/sub-S02/ses-E02/mod-mr/anat/sub-S02_ses-E02_Sag_T1.nii.gz ===

1. Dimensiones de la matriz (x, y, z, t): (432, 432, 12)
2. Resolución (Tamaño del vóxel en mm): (np.float32(0.625), np.float32(0.625), np.float32(4.5))
3. Matriz Afín (Muestra de la orientación):
[[-8.49733187e-04 -4.58108727e-03  4.49987507e+00]
 [-6.24999404e-01 -1.90343853e-05 -6.11924799e-03]
 [ 2.52633690e-05 -6.24983191e-01 -3.29836123e-02]]
4. Tipo de dato numérico: float32

=== ESTADÍSTICAS DE SEÑAL ===
Intensidad Mínima: 0.0
Intensidad Máxima: 2730.065673828125 (Si es 0, la imagen está negra)
Intensidad Media:  421.03
